# 01 — estimating the information-flow network

who moves first? we estimate a directed network between ETFs using transfer entropy
(with a permutation test) and attach granger p-values as a second opinion.

run `python run.py` for the full pipeline — this notebook is just the interactive tour.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from diffuse import data as D
from diffuse import network as N

# small universe so this runs in a couple of minutes
prices = D.fetch_prices(D.UNIVERSE[:10], interval="1h", period="1mo")
D.validate_panel(prices)
rets = D.to_returns(prices)
rets.shape

In [ ]:
# sanity: TE should be ~0 for independent series, >0 when coupled
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=2000)
y = np.concatenate([[0], 0.6 * x[:-1]]) + rng.normal(size=2000) * 0.5  # x leads y
print("TE x->y:", round(N.transfer_entropy(x, y), 4))
print("TE y->x:", round(N.transfer_entropy(y, x), 4))

In [ ]:
G = N.build_network(rets, n_shuffles=100)
print("leaders:", N.top_leaders(G, k=5))

In [ ]:
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(9, 7))
nx.draw(G, pos, with_labels=True, node_size=800, alpha=0.8,
        width=[1 + 5 * d["te"] for _, _, d in G.edges(data=True)],
        arrowsize=14, node_color="steelblue", font_size=9)
plt.title("info-flow network, edge width = transfer entropy")
plt.show()

## what to look at

- SPY/QQQ usually sit at the center with high out-strength — the market factor moves first, sectors follow. that's expected, not alpha.
- the interesting edges are the *sector-to-sector* ones (e.g. XLE -> XLI). those are the candidates the backtest trades on.
- if the graph is nearly empty, the permutation test is doing its job — most pairwise "lead-lag" you see in raw correlations doesn't survive.